In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [22]:
cc_trips = pd.read_parquet('../data/processed/cleaned_credit_card_trips.parquet')
cc_trips.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_hour,pickup_day,tip_percentage
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,3.66,0.0,1.0,15.86,2.5,0.0,0.00,0,Thursday,50.83
1,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,11.11,0.0,1.0,55.56,2.5,0.0,0.75,0,Thursday,28.71
2,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,...,4.99,0.0,1.0,24.94,2.5,0.0,0.75,0,Thursday,35.14
3,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,...,8.61,0.0,1.0,51.66,2.5,0.0,0.75,0,Thursday,23.08
4,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3.0,1.83,1.0,N,237,263,1,...,2.36,0.0,1.0,18.06,2.5,0.0,0.00,0,Thursday,22.06


### Target Creation & Class Imbalance

In [23]:
# Determining generous tip
cc_trips['is_generous_tip'] = (cc_trips['tip_percentage'] > 20).astype(int)
cc_trips['is_generous_tip']

# Generous tip distribution
generous_dist = cc_trips['is_generous_tip'].value_counts(normalize = True)*100
generous_dist.head()

is_generous_tip
1    72.920279
0    27.079721
Name: proportion, dtype: float64

### Feature Engineering, Splitting, and Scaling

In [24]:
# Duration in minutes
cc_trips['trip_duration'] = (cc_trips['tpep_dropoff_datetime'] - cc_trips['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Defining features and target
features = ['trip_distance', 'fare_amount', 'trip_duration', 'pickup_hour']
X = cc_trips[features]
y = cc_trips['is_generous_tip']

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

### Model Training, Evaluation, and Business Insight

In [25]:
# model training
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Prediction
y_pred = model.predict(X_test_scaled)

# evaluation
print(f'Accuracy Score {accuracy_score(y_test, y_pred)}')
print(f'Classification Report {classification_report(y_test, y_pred)}')

Accuracy Score 0.7493925774998986
Classification Report               precision    recall  f1-score   support

           0       0.68      0.13      0.22    119603
           1       0.75      0.98      0.85    324075

    accuracy                           0.75    443678
   macro avg       0.72      0.55      0.54    443678
weighted avg       0.73      0.75      0.68    443678

